# How to solve a general eigenvalue problem

Step 1: Create params object


In [9]:
from    FELiCS.Parameters.config            import config    # Read parameters

settingsFileName = "modal.json"
param       = config()
param.importFromFile(settingsFileName)

mesh        = param.getMesh()

Info     | config.py              | importFromFile             (line 170 ) : Loading configuration from modal.json
Warning  | config.py              | importFromFile             (line 189 ) : Field "MolViscPerturbModel" missing from file, setting default: {'type': 'Constant', 'Constants': {'Viscosity': 1.0}}
Warning  | config.py              | importFromFile             (line 189 ) : Field "PrandtlNumber" missing from file, setting default: 0.72
Warning  | config.py              | importFromFile             (line 189 ) : Field "ForcingCoeff" missing from file, setting default: []
Warning  | config.py              | importFromFile             (line 189 ) : Field "ForcingNorm" missing from file, setting default: TKE
Warning  | config.py              | importFromFile             (line 189 ) : Field "ResponseNorm" missing from file, setting default: TKE
Info     | logging.py             | change_log_location        (line 263 ) : Log files moved to: output_dir/log
Info     | MixtureClass.py

In [ ]:
import numpy as np

from  FELiCS.SpaceDisc.FEMSpaces    import FEMSpaces
from    FELiCS.Fields.meanFlowClass         import meanFlowClass
from    FELiCS.Equation.EquationCollection  import EquationCollectionClass
from    FELiCS.Fields.Field                  import Field

# Get FEMSpaces
FEMS   = FEMSpaces(
    param,
    mesh,
)

# Initialize mean flow class & import from file
meanFlow    = meanFlowClass(
    param, 
    FEMS, 
    mesh
)
meanFlow.importDataFromFileAndExportToH5()


# baseFlow   = Field(FEMSpaces.VMixed, mesh)
baseFlow_array = meanFlow._fieldDict['u'].function.x.array[:]

# Load base flow into meanFlow object
if np.linalg.norm(meanFlow._fieldDict['u'].function.x.array[:]) < 1.e-8:
    baseFlow    = Field(FEMS.VMixed, mesh)
    baseFlow.setCoefficientArray(baseFlow_array)
    [u,p]       = baseFlow.getListOfSingleFields()
    meanFlow._fieldDict['u'] = u.function

# Export mean flow in "h5" file
meanflowFilename = 'meanflow.h5'
meanFlow.mapToExportMeshAndExport(FEMS, meanflowFilename)

# Set up quations
equation    = EquationCollectionClass(
    param,
    FEMS,
    meanFlow,
    mesh
)


Info     | FEMSpaces.py           | __init__                   (line 90  ) : Defining FEM-spaces.
Info     | meanFlowClass.py       | importDataFromFileAndExportToH5 (line 56  ) : Reading input flow from: 'base_flow_for_FELiCS.fel'
Info     | meanFlowClass.py       | InterpolateOnFELiCSMesh    (line 389 ) : Interpolating mean flow on FELiCS mesh.
Warning  | meanFlowClass.py       | raiseNotInFileListWarning  (line 559 ) : Field 'rho' not in import file. Assuming Field is one.
Warning  | meanFlowClass.py       | raiseNotInFileListWarning  (line 561 ) : Field 'spg' not in import file. Assuming Field is zero.
Info     | meanFlowClass.py       | mapToExportMeshAndExport   (line 146 ) : Mapping mean flow to export mesh and exporting.
Info     | meanFlowClass.py       | mapToExportMeshAndExport   (line 146 ) : Mapping mean flow to export mesh and exporting.
Info     | EquationCollection.py  | __init__                   (line 119 ) : Initializing the equation collection class.


In [ ]:
from    FELiCS.Solvers.LinearSolver         import LinearSolver
from    FELiCS.Fields.ModeCollection        import ModeCollection

# Get matrices for eigenproblem
A       = equation.getLinearOperator(meanFlow)
B       = equation.getWeightMatrix  (meanFlow)

guesses  = param.Numerics.EigenValueGuess
nSol    = param.Numerics.nSolut
# Solve direct eigenproblem for each guess
solution    = ModeCollection(
    FEMSpaces.VMixed, 
    mesh
)
for guess in guesses:
    tmp = LinearSolver.solveGeneralEigenproblem(
        A,
        B,
        guess,
        nSol,
    )
    solution.appendSolutionOfEigenProblem(tmp, guess)